# 1. IMPORTS & SYNTHETIC DATASET CREATION

In [1]:


import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler


In [2]:

# Ek sample dataset banate hain jisme alag-alag tarah ke raw features hon
np.random.seed(42)
data = pd.DataFrame(
    {
        "Age": [25, 30, np.nan, 45, 22, 38, np.nan, 50],
        "Salary": [50000, 60000, 55000, 120000, 48000, np.nan, 80000, 110000],
        "City": [
            "Karachi",
            "Lahore",
            "Islamabad",
            "Karachi",
            "Lahore",
            "Karachi",
            "Islamabad",
            "Peshawar",
        ],
        "Education": [
            "Bachelors",
            "Masters",
            "Bachelors",
            "PhD",
            "High School",
            "Masters",
            "High School",
            "PhD",
        ],
        "Review": [
            "Great product excellent service",
            "Bad quality highly disappointed",
            "Average performance ok product",
            "Excellent product highly recommended",
            "Worst quality service bad",
            "Good quality fast delivery",
            "Decent experience average quality",
            "Amazing excellent product",
        ],
        "Joined_Date": pd.date_range(start="2022-01-01", periods=8, freq="ME"),
        "Target_Buy": [0, 1, 0, 1, 0, 1, 0, 1],
    }
)

print("--- Original Raw Dataset ---")
print(data)



--- Original Raw Dataset ---
    Age    Salary       City    Education  \
0  25.0   50000.0    Karachi    Bachelors   
1  30.0   60000.0     Lahore      Masters   
2   NaN   55000.0  Islamabad    Bachelors   
3  45.0  120000.0    Karachi          PhD   
4  22.0   48000.0     Lahore  High School   
5  38.0       NaN    Karachi      Masters   
6   NaN   80000.0  Islamabad  High School   
7  50.0  110000.0   Peshawar          PhD   

                                 Review Joined_Date  Target_Buy  
0       Great product excellent service  2022-01-31           0  
1       Bad quality highly disappointed  2022-02-28           1  
2        Average performance ok product  2022-03-31           0  
3  Excellent product highly recommended  2022-04-30           1  
4             Worst quality service bad  2022-05-31           0  
5            Good quality fast delivery  2022-06-30           1  
6     Decent experience average quality  2022-07-31           0  
7             Amazing excellent produ

In [5]:
data.isnull().sum()

Age               2
Salary            1
City              0
Education         0
Review            0
Joined_Date       0
Target_Buy        0
Age_Imputed       0
Salary_Imputed    0
dtype: int64

In [6]:


# 2. MISSING VALUE IMPUTATION
# Numerical columns me missing values ko median se fill karte hain
data["Age_Imputed"] = data["Age"].fillna(data["Age"].median())



In [ ]:


# 3. FEATURE TRANSFORMATION & SCALING
# StandardScaler: Mean = 0, Std = 1
scaler_std = StandardScaler()
data["Salary_Standardized"] = scaler_std.fit_transform(
    data[["Salary_Imputed"]]
)


In [8]:

# MinMaxScaler: Range [0, 1]
scaler_minmax = MinMaxScaler()
data["Age_MinMax"] = scaler_minmax.fit_transform(data[["Age_Imputed"]])



In [ ]:

# 4. CATEGORICAL ENCODING
# A. Ordinal Encoding (Jahan order/rank matter karta hai)
education_map = {"High School": 1, "Bachelors": 2, "Masters": 3, "PhD": 4}
data["Education_Encoded"] = data["Education"].map(education_map)


In [10]:

# B. One-Hot Encoding (Nominal categories ke liye)
ohe = OneHotEncoder(sparse_output=False, drop="first")
city_encoded = ohe.fit_transform(data[["City"]])
city_encoded_df = pd.DataFrame(
    city_encoded, columns=ohe.get_feature_names_out(["City"])
)



In [11]:

# ==========================================
# 5. DATETIME FEATURE EXTRACTION
# ==========================================
data["Join_Year"] = data["Joined_Date"].dt.year
data["Join_Month"] = data["Joined_Date"].dt.month
data["Join_DayOfWeek"] = data["Joined_Date"].dt.dayofweek



In [12]:

# ==========================================
# 6. TEXT FEATURE ENGINEERING (TF-IDF)
# ==========================================
tfidf = TfidfVectorizer(max_features=3)
tfidf_matrix = tfidf.fit_transform(data["Review"]).toarray()
tfidf_df = pd.DataFrame(
    tfidf_matrix,
    columns=[f"tfidf_{word}" for word in tfidf.get_feature_names_out()],
)



In [13]:

# ==========================================
# 7. FEATURE CREATION (Interaction Features)
# ==========================================
# Ek naya meaningful ratio feature: Salary to Age Ratio
data["Salary_Per_Age"] = data["Salary_Imputed"] / data["Age_Imputed"]



In [14]:

# ==========================================
# 8. FINAL CONSOLIDATED DATASET
# ==========================================
processed_df = pd.concat([data, city_encoded_df, tfidf_df], axis=1)

# Direct raw columns ko drop karke sirf engineered features rakhein
columns_to_drop = [
    "Age",
    "Salary",
    "City",
    "Education",
    "Review",
    "Joined_Date",
]
final_features = processed_df.drop(columns=columns_to_drop)

print("\n--- Final Processed Features for ML Model ---")
print(final_features.head())


--- Final Processed Features for ML Model ---
   Target_Buy  Age_Imputed  Salary_Imputed  Salary_Standardized  Age_MinMax  \
0           0         25.0         50000.0            -0.876984    0.107143   
1           1         30.0         60000.0            -0.493603    0.285714   
2           0         34.0         55000.0            -0.685294    0.428571   
3           1         45.0        120000.0             1.806684    0.821429   
4           0         22.0         48000.0            -0.953661    0.000000   

   Education_Encoded  Join_Year  Join_Month  Join_DayOfWeek  Salary_Per_Age  \
0                  2       2022           1               0     2000.000000   
1                  3       2022           2               0     2000.000000   
2                  2       2022           3               3     1617.647059   
3                  4       2022           4               5     2666.666667   
4                  1       2022           5               1     2181.818182   

   